# Part 3.1 Internal Bottom-up Baseline

Notebook này chứa chính baseline nội bộ được tái dựng từ official dataset.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path('/home/lducc/code/datathon/datathon-2026-round-1')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.data import load_dataframes
from models.baseline_internal_bottomup import build_internal_bottomup_baseline, _monthly_components, _seasonal_trend_forecast, _daily_shape_profile
from models.final_shape_medium import build_final_shape_medium_submission
from scripts.run_part2_pipeline import configure_style, plot_final_shape_medium_monthly_revenue, plot_final_shape_medium_correction
from models.data import build_daily_frame

configure_style()
data = load_dataframes()
daily = build_daily_frame(data)


In [2]:
monthly = _monthly_components(daily)
monthly[['date', 'orders_per_day', 'rpo', 'cogs_ratio']].tail()


,date,orders_per_day,rpo,cogs_ratio
121,2022-08-01,105.612903,34680.190431,0.828175
122,2022-09-01,88.100000,32449.597650,0.914078
123,2022-10-01,67.645161,35858.107921,0.824129
124,2022-11-01,56.766667,30651.838896,0.902260
125,2022-12-01,77.419355,21856.223417,1.019726


In [3]:
future_months = pd.period_range('2023-01', '2024-06', freq='M').to_timestamp()
orders_fc = _seasonal_trend_forecast(monthly, 'orders_per_day', future_months, floor=100.0)
rpo_fc = _seasonal_trend_forecast(monthly, 'rpo', future_months, floor=1000.0)
ratio_fc = _seasonal_trend_forecast(monthly, 'cogs_ratio', future_months, floor=0.78, cap=0.98)
orders_fc.head()


2023-01-01    100.000000
2023-02-01    100.000000
2023-03-01    132.271077
2023-04-01    156.171869
2023-05-01    140.457453
Freq: MS, dtype: float64

In [4]:
shape = _daily_shape_profile(daily, pd.date_range('2023-01-01', '2024-06-30', freq='D'))
shape.head()


2023-01-01    0.053785
2023-01-02    0.023509
2023-01-03    0.014699
2023-01-04    0.020608
2023-01-05    0.018771
dtype: float64

In [5]:
outputs = build_internal_bottomup_baseline()
pd.read_csv(outputs['baseline_submission']).head()


,Date,Revenue,COGS
0,2023-01-01,5822207.02,4866599.83
1,2023-01-02,2544788.43,2127108.65
2,2023-01-03,1591204.54,1330037.86
3,2023-01-04,2230773.67,1864633.59
4,2023-01-05,2031893.80,1698396.16


**Diễn giải**

Baseline này forecast theo component orders/day, revenue per order và cogs ratio. Nó học trend và seasonality từ lịch sử chính thức, không dùng external anchor và không hand-set growth.
